<div dir="rtl">

# نوت‌بوک ۲: ممیزی کیفیت داده

**فاز ۱، تسک ۵**

سند برای ممیزی کیفیت پنج چیز می‌خواهد و برای هر کدام یک **تصمیم اولیه با دلیل**:

۱. مقادیر گمشده
۲. رکوردهای تکراری
۳. مقادیر ناممکن
۴. ناسازگاری نوع داده
۵. نمونه‌های پرت

این نوت‌بوک **چیزی پاک نمی‌کند**؛ فقط مشکلات را می‌شمارد و تصمیم را ثبت می‌کند. اجرای تصمیم‌ها (پاک‌سازی واقعی) در فاز ۲ انجام می‌شود. جدول نهایی تصمیم‌ها در `reports/phase1/quality_decisions.csv` ذخیره می‌شود.

</div>

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load(open(ROOT / "configs" / "config.yaml", encoding="utf-8"))
PROC = ROOT / cfg["data"]["processed_dir"]
FIG = ROOT / "figures"
REP = ROOT / "reports" / "phase1"

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

reviews = pd.read_parquet(PROC / "reviews_rc.parquet")
meta = pd.read_parquet(PROC / "meta_rc.parquet")
N, M = len(reviews), len(meta)
print(f"reviews: {N:,}   products: {M:,}")

def pct(n, total):
    return f"{n:,} ({100 * n / total:.2f}%)"

<div dir="rtl">

## ۱. مقادیر گمشده

«گمشده» فقط خانه‌ی خالی (null) نیست. یک متن خالی (`""`) یا یک لیست خالی (`[]`) هم در عمل یعنی «اطلاعاتی نداریم». برای همین سه حالت را جدا می‌شماریم.

</div>

In [ ]:
def missing_table(df, name):
    rows = []
    for c in df.columns:
        s = df[c]
        null = s.isna().sum()
        empty = 0
        if s.dtype == object or str(s.dtype).startswith("str"):
            t = s.dropna().astype(str).str.strip()
            empty = t.isin(["", "[]", "{}"]).sum()
        rows.append({"file": name, "column": c,
                     "null": null, "empty": empty,
                     "missing_total_%": round(100 * (null + empty) / len(df), 2)})
    return pd.DataFrame(rows)

miss = pd.concat([missing_table(reviews, "reviews"), missing_table(meta, "meta")])
miss[miss["missing_total_%"] > 0].sort_values("missing_total_%", ascending=False)

<div dir="rtl">

توجه: ستون `images` در نقدها برای بیشتر نقدها لیست خالی است؛ این یعنی «کاربر عکسی نگذاشته»، نه «داده گم شده». تحلیل دسترس‌پذیری هر وجه (متن، تصویر، قیمت) در تسک ۶ می‌آید.

</div>

<div dir="rtl">

## ۲. رکوردهای تکراری

تکراری بودن چند سطح دارد و هر سطح معنای متفاوتی دارد:

| سطح | تعریف | معنی احتمالی |
|---|---|---|
| الف | همه‌ی ستون‌ها یکسان | یک نقد دو بار در فایل آمده؛ خطای جمع‌آوری |
| ب | همان کاربر، همان محصول، همان زمان | همان الف، با کلید کوچک‌تر |
| ج | همان کاربر، همان محصول، زمان متفاوت | کاربر دو نقد برای یک محصول نوشته (مثلاً به‌روزرسانی) |
| د | همان کاربر، همان متن، محصول متفاوت | یک متن برای چند محصول کپی شده |
| هـ | کاربران متفاوت، متن یکسان و بلند | متن قالبی، تبلیغاتی یا اسپم |

</div>

In [ ]:
long_text = reviews["text"].str.len() > 50

dup = pd.Series({
    "A  all columns identical":            reviews.duplicated().sum(),
    "B  same user + product + time":       reviews.duplicated(["user_id", "parent_asin", "timestamp"]).sum(),
    "C  same user + product (any time)":   reviews.duplicated(["user_id", "parent_asin"]).sum(),
    "D  same user + text, other product":  (reviews.duplicated(["user_id", "text"])
                                            & ~reviews.duplicated(["user_id", "parent_asin", "text"])).sum(),
    "E  same long text, any user":         reviews[long_text].duplicated("text").sum(),
})
dup.map(lambda n: pct(n, N))

<div dir="rtl">

نمونه‌ای از سطح «د» (یک کاربر، یک متن، چند محصول) و سطح «هـ» (یک متن، چند کاربر):

</div>

In [ ]:
d_mask = reviews.duplicated(["user_id", "text"], keep=False) & (reviews["text"].str.len() > 30)
example_user = reviews.loc[d_mask, "user_id"].iloc[0]
cols = ["user_id", "parent_asin", "asin", "rating", "text"]
print("Level D example:")
display(reviews.loc[d_mask & (reviews["user_id"] == example_user), cols].head(4))

e_counts = reviews.loc[long_text, "text"].value_counts()
print("Level E: most repeated long texts")
e_counts.head(5).rename("times").to_frame()

<div dir="rtl">

## ۳. مقادیر ناممکن

برای هر ستون عددی یا زمانی یک بازه‌ی «ممکن» تعریف می‌کنیم و بیرون از آن را می‌شماریم.

</div>

In [ ]:
ts = pd.to_datetime(reviews["timestamp"], unit="ms", utc=True)
collection_end = pd.Timestamp("2023-09-30", tz="UTC")

impossible = pd.Series({
    "rating outside 1..5":                 (~reviews["rating"].between(1, 5)).sum(),
    "rating not a whole number":           (reviews["rating"] % 1 != 0).sum(),
    "helpful_vote < 0":                    (reviews["helpful_vote"] < 0).sum(),
    "review before 1996 (Amazon reviews)": (ts < pd.Timestamp("1996-01-01", tz="UTC")).sum(),
    "review after data collection":        (ts > collection_end).sum(),
    "price <= 0":                          (meta["price"] <= 0).sum(),
    "average_rating outside 1..5":         (~meta["average_rating"].between(1, 5)).sum(),
    "rating_number < 1":                   (meta["rating_number"] < 1).sum(),
})
impossible.to_frame("count")

In [ ]:
reviews["rating"].value_counts().sort_index().to_frame("reviews").assign(
    share_pct=lambda d: (100 * d["reviews"] / N).round(1))

<div dir="rtl">

امتیازها همه عدد صحیح ۱ تا ۵اند. توزیع آن‌ها **J شکل** است: بیشترین سهم با ۵ ستاره، بعد ۱ ستاره، و وسط خالی. این همان سوگیری خودگزینی است که در تسک ۲ پیش‌بینی کرده بودیم.

</div>

<div dir="rtl">

## ۴. ناسازگاری نوع داده

این بررسی در نوت‌بوک ۱ (بخش ۲) روی کل فایل خام انجام شد:

- در فایل نقدها هیچ ناسازگاری نوعی نبود.
- در فایل اطلاعات محصول، `price` همیشه عدد یا خالی است و هیچ قیمت متنی دیده نشد.
- `author` و `subtitle` تقریباً در هیچ محصولی وجود ندارند.

اینجا فقط یک بررسی تکمیلی می‌کنیم: آیا در متن نقدها **نشانه‌گذاری فنی** (مثل تگ HTML یا کد ویدیو) هست که متن را آلوده کند؟

</div>

In [ ]:
patterns = {
    "HTML tag (<br /> etc.)": r"<[a-zA-Z/][^>]*>",
    "embedded video code [[VIDEOID:...]]": r"\[\[VIDEOID:[^\]]*\]\]",
    "HTML entity (&amp; etc.)": r"&[a-z]+;|&#\d+;",
    "URL": r"https?://",
}
has = pd.DataFrame({name: reviews["text"].str.contains(p, regex=True, na=False)
                    for name, p in patterns.items()})
artifacts = has.sum()
any_markup = has.any(axis=1).sum()   # a review can contain several kinds
print(artifacts.map(lambda n: pct(n, N)).to_string())
print()
print("reviews with ANY of the above:", pct(any_markup, N))

<div dir="rtl">

## ۵. نمونه‌های پرت

پرت یعنی مقداری که خیلی با بقیه فرق دارد. پرت **لزوماً غلط نیست**؛ ممکن است یک مقدار واقعی ولی کمیاب باشد. برای همین برای هر مورد اول نگاه می‌کنیم و بعد تصمیم می‌گیریم.

### ۵-۱. طول متن نقد

</div>

In [ ]:
text_len = reviews["text"].str.strip().str.len()
reviews["text_len"] = text_len

length_bins = pd.Series({
    "empty (0 chars)":     (text_len == 0).sum(),
    "1-3 chars":           text_len.between(1, 3).sum(),
    "4-20 chars":          text_len.between(4, 20).sum(),
    "21-5000 chars":       text_len.between(21, 5000).sum(),
    "over 5000 chars":     (text_len > 5000).sum(),
})
print(length_bins.map(lambda n: pct(n, N)).to_string())
print()
print("examples of 1-20 char reviews:")
print(reviews.loc[text_len.between(1, 20), "text"].value_counts().head(10).to_string())

<div dir="rtl">

### ۵-۲. رأی «مفید بود»

</div>

In [ ]:
hv = reviews["helpful_vote"]
print(hv.describe(percentiles=[.5, .9, .99, .999]).round(1).to_string())
print()
print("most voted reviews:")
reviews.nlargest(3, "helpful_vote")[["helpful_vote", "rating", "text_len", "parent_asin"]]

<div dir="rtl">

### ۵-۳. قیمت

</div>

In [ ]:
p = meta["price"].dropna()
print(p.describe(percentiles=[.01, .5, .99]).round(2).to_string())
print()
print("products over $1000:", (p > 1000).sum())
meta.loc[meta["price"] > 1000, ["price", "title"]].sort_values("price", ascending=False).head(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(p, bins=80, color="#4C72B0")
axes[0].set_title("Price (linear scale)")
axes[0].set_xlabel("USD")
axes[1].hist(np.log10(p), bins=80, color="#4C72B0")
axes[1].set_title("Price (log10 scale)")
axes[1].set_xlabel("log10(USD)")
for ax in axes:
    ax.set_ylabel("products")
plt.tight_layout()
plt.savefig(FIG / "fig02_price_outliers.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

در جدول بالا، بیشتر محصولات گران واقعاً محصولات کنترلی گران‌قیمت‌اند (مثل پهپاد DJI Inspire)، ولی **یک ترازوی دقیق** هم در میانشان هست. یعنی درخت دسته‌بندی آمازون گاهی محصولی را اشتباه در این شاخه گذاشته است؛ همان مشکلی که در نوت‌بوک ۱ با کابل HDMI دیدیم.

در مقیاس خطی تقریباً همه‌ی محصولات در یک ستون کنار صفر جمع شده‌اند و چند محصول گران همه‌ی نمودار را کش داده‌اند. در **مقیاس لگاریتمی** توزیع خوانا می‌شود. نتیجه: قیمت را در فاز ۲ با لگاریتم وارد می‌کنیم، نه با حذف پرت‌ها.

### ۵-۴. کاربران پرنقد

</div>

In [ ]:
per_user = reviews.groupby("user_id").agg(
    reviews=("rating", "size"),
    mean_rating=("rating", "mean"),
    verified_share=("verified_purchase", "mean"),
    median_text_len=("text_len", "median"),
    first=("timestamp", "min"),
    last=("timestamp", "max"),
)
for c in ("first", "last"):
    per_user[c] = pd.to_datetime(per_user[c], unit="ms").dt.year

overall = pd.DataFrame({
    "reviews": [1], "mean_rating": [reviews["rating"].mean()],
    "verified_share": [reviews["verified_purchase"].mean()],
    "median_text_len": [text_len.median()], "first": [None], "last": [None],
}, index=["(all reviewers)"])

pd.concat([per_user.nlargest(10, "reviews"), overall]).round(2)

<div dir="rtl">

اگر کاربران پرنقد **اسپم** بودند، انتظار داشتیم همه‌ی نقدهایشان ۵ ستاره، کوتاه و در یک بازه‌ی زمانی کوتاه باشد. جدول بالا را با این انتظار مقایسه کنید: بیشتر کاربران پرنقد امتیازهای متنوع و متن‌های بلند در چند سال دارند (شبیه علاقه‌مندان جدی)، ولی چند نفر الگوی مشکوک دارند: متن‌های چند کاراکتری، امتیاز بالا و فعالیت در بازه‌ای کوتاه.

</div>

<div dir="rtl">

## ۶. جدول تصمیم‌ها

برای هر مشکل: چقدر است، چه تصمیمی گرفتیم و چرا. این جدول در گزارش فاز ۱ می‌آید.

</div>

In [ ]:
d_level = (reviews.duplicated(["user_id", "text"])
           & ~reviews.duplicated(["user_id", "parent_asin", "text"])).sum()
e_level = reviews[long_text].duplicated("text").sum()

decisions = pd.DataFrame([
    ("Exact duplicate reviews (level A/B)", dup.iloc[0],
     "drop (keep first copy)",
     "same user, product and second: a collection error, not a second opinion"),
    ("Same user + product, different time (level C)", dup.iloc[2] - dup.iloc[1],
     "keep, flag",
     "can be a genuine update after using the product longer"),
    ("Same user + text on another product (level D)", d_level,
     "keep, flag; revisit in phase 2",
     "may be one review shown on sister listings; would double-count text when learning aspects"),
    ("Same long text from different users (level E)", e_level,
     "keep, flag; inspect in phase 2",
     "may be templates or spam; too early to delete without reading them"),
    ("Empty review text", (text_len == 0).sum(),
     "keep for rating analysis; no text features",
     "the star rating is still valid information"),
    ("Very short text (1-20 chars)", text_len.between(1, 20).sum(),
     "keep",
     "'Great!' is a real opinion; it simply carries no aspect information"),
    ("Reviews with markup in text (HTML, video codes)", int(any_markup),
     "clean in phase 2 preprocessing",
     "markup is not customer language and would confuse text models"),
    ("Impossible values (rating, time, price)", int(impossible.sum()),
     "none found; no action", "all values inside valid ranges"),
    ("Price missing", meta["price"].isna().sum(),
     "keep; study in task 6",
     "need to know if missingness is random before choosing how to handle it"),
    ("Price outliers (> $1000)", (meta["price"] > 1000).sum(),
     "keep; use log(price); flag miscategorised items",
     "mostly real high-end products (e.g. DJI Inspire drones); a few are miscategorised (e.g. a weighing scale)"),
    ("helpful_vote outliers", (reviews["helpful_vote"] > reviews["helpful_vote"].quantile(.999)).sum(),
     "keep; never a model feature",
     "real but collected after the review was written (leakage)"),
    ("Heavy reviewers (>= 50 reviews)", int((per_user["reviews"] >= 50).sum()),
     "keep, flag; inspect suspicious ones in phase 2",
     "most look like enthusiasts (mixed ratings, long texts, many years); a few look suspicious (very short texts, high ratings, short active period)"),
], columns=["issue", "count", "decision", "reason"])

decisions.to_csv(REP / "quality_decisions.csv", index=False)
decisions

<div dir="rtl">

## ۷. جمع‌بندی

_این بخش را بعد از اجرای نوت‌بوک و دیدن خروجی‌ها با هم می‌نویسیم._

</div>